In [ ]:
import joblib

import numpy as np
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer ,PorterStemmer
import spacy

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB  
from hmmlearn.hmm import GaussianHMM

# from sktime.detection.hmm_learn import GaussianHMM 

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.decomposition import TruncatedSVD #instead of PCA
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline  



In [2]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer", "tagger"])

In [3]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
stop_words=set(stopwords.words('english'))

# Load the data + class Weights

In [ ]:
data = joblib.load('news_data.pkl') # original
x_train = data['X_train']
y_train = data['y_train']
x_test = data['X_test']
y_test = data['y_test']

data_balance = joblib.load('news_data_resampled.pkl') # oversample only
x_train_ran_res = data_balance['X_train']
y_train_ran_res = data_balance['y_train']
x_test_ran_res = data_balance['X_test']
y_test_ran_res = data_balance['y_test']

data_balance_rosrus=joblib.load('news_data_bal_ros_rus.pkl')  #rosrus
x_train_bal = data_balance_rosrus['X_train']
y_train_bal = data_balance_rosrus['y_train']
x_test_bal = data_balance_rosrus['X_test']
y_test_bal = data_balance_rosrus['y_test']

In [6]:
data_undersampled=joblib.load('news_data_undersampled.pkl') # undersample only
x_train_undersampled=data_undersampled['X_train']
y_train_undersampled=data_undersampled['y_train']
x_test_undersampled=data_undersampled['X_test']
y_test_undersampled=data_undersampled['y_test']

In [ ]:
original_df_train=pd.DataFrame({"text":x_train})
original_df_test=pd.DataFrame({"text":x_test})

undersample_df_train=pd.DataFrame({"text":x_train_undersampled})
undersample_df_test=pd.DataFrame({"text":x_test_undersampled})

rosrus_df_train=pd.DataFrame({"text":x_train_bal})
rosrus_df_test=pd.DataFrame({"text":x_test_bal})

In [7]:
class_weights_dict=joblib.load('classWeightsDic')

# Initial  NER+POS

Stopword removal was intentionally not applied to the NER pipeline 
Because NER depends on full contextual information. Stopwords are part of the sentence structure and removing them can break entity boundaries and degrade recognition accuracy.

In [ ]:
def NER_features(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    doc = nlp(text)
    return [ent.label_ for ent in doc.ents]

    # tokens = word_tokenize(text.lower())
    # pos_tags = nltk.pos_tag(tokens)
    # ne_tree = nltk.ne_chunk(pos_tags, binary=False)

    # ner_tokens = []

    # for subtree in ne_tree:
    #     if hasattr(subtree, 'label'):
    #         ner_tokens.append(subtree.label())

    # return ' '.join(ner_tokens)

In [9]:
def pos_features(text, remove_stopwords=False):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text.lower())

    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]

    pos_tags = nltk.pos_tag(tokens)
    pos_tokens = [tag for _, tag in pos_tags]
    return pos_tokens


In [10]:
num_classes = len(set(y_train))  
num_classes

10

# Original Data

In [11]:
len(x_train)

167616

## POS_NER feature Extraction + scalling

In [ ]:
original_df_train['POS']=original_df_train['text'].apply(pos_features)
original_df_test['POS']=original_df_test['text'].apply(pos_features)
original_df_train['NER']=original_df_train['text'].apply(NER_features)
original_df_test['NER']=original_df_test['text'].apply(NER_features)

In [ ]:
original_df_train['POS+NER']=original_df_train['POS']+original_df_train['NER']
original_df_test['POS+NER']=original_df_test['POS']+original_df_test['NER']

In [180]:
original_df_train

,text,POS,NER,POS+NER,POS+NER_str
69047,Faith Groups Are Rallying Against North Caroli...,"[NN, NNS, VBP, VBG, IN, JJ, JJ, NN, NN]",[],"[NN, NNS, VBP, VBG, IN, JJ, JJ, NN, NN]",NN NNS VBP VBG IN JJ JJ NN NN
52038,Man Dresses Up Like Cat's Favorite Toy And It ...,"[NN, VBZ, RP, IN, NNS, VBP, NN, CC, PRP, VBZ, ...",[],"[NN, VBZ, RP, IN, NNS, VBP, NN, CC, PRP, VBZ, ...",NN VBZ RP IN NNS VBP NN CC PRP VBZ RB VB RB
92092,Taylor Swift Sends Love To A Fan After Emotion...,"[NN, NN, VBZ, VBP, TO, DT, NN, IN, JJ, NN, NN]",[PERSON],"[NN, NN, VBZ, VBP, TO, DT, NN, IN, JJ, NN, NN,...",NN NN VBZ VBP TO DT NN IN JJ NN NN PERSON
15791,Homeless Will Now Be Asked: Are You Fleeing Do...,"[NN, MD, RB, VB, VBN, VBP, PRP, VBG, JJ, NN]",[],"[NN, MD, RB, VB, VBN, VBP, PRP, VBG, JJ, NN]",NN MD RB VB VBN VBP PRP VBG JJ NN
50596,Someone Stuffed Canadian Mailboxes With Anti-C...,"[NN, VBD, JJ, NNS, IN, JJ, NNS, IN, JJ, NNS, NN]",[PERSON],"[NN, VBD, JJ, NNS, IN, JJ, NNS, IN, JJ, NNS, N...",NN VBD JJ NNS IN JJ NNS IN JJ NNS NN PERSON
...,...,...,...,...,...
151462,"Pointers, Please, for Spotting Wild Life: Mr. ...","[NNS, VBP, IN, VBG, JJ, NN, NN, NN, IN, DT, NN...","[PERSON, ORG]","[NNS, VBP, IN, VBG, JJ, NN, NN, NN, IN, DT, NN...",NNS VBP IN VBG JJ NN NN NN IN DT NN NN PERSON ORG
152002,Corporate Stress on Young Hearts: A Reflection,"[JJ, NN, IN, JJ, NNS, DT, NN]",[],"[JJ, NN, IN, JJ, NNS, DT, NN]",JJ NN IN JJ NNS DT NN
1961,MSNBC Hosts Crack Up Over Error Found On First...,"[JJ, NNS, VBP, RP, IN, NN, VBN, IN, JJ, NN, IN...",[ORG],"[JJ, NNS, VBP, RP, IN, NN, VBN, IN, JJ, NN, IN...",JJ NNS VBP RP IN NN VBN IN JJ NN IN NN NN ORG
47298,"On Balance, 2016 Was A Pretty Garbage Year For...","[IN, NN, VBD, DT, RB, JJ, NN, IN, JJ, NN]",[],"[IN, NN, VBD, DT, RB, JJ, NN, IN, JJ, NN]",IN NN VBD DT RB JJ NN IN JJ NN


In [187]:
original_df_train['text'][92092]

'Taylor Swift Sends Love To A Fan After Emotional Tumblr Post'

In [188]:
original_df_train['NER'][92092]

['PERSON']

In [189]:
original_df_train['POS+NER'][92092]

['NN', 'NN', 'VBZ', 'VBP', 'TO', 'DT', 'NN', 'IN', 'JJ', 'NN', 'NN', 'PERSON']

In [21]:
original_df_test

,text,POS,NER,POS+NER
28004,Kuwait Reportedly Deports 76 Gay Men In Crackdown,"[NN, RB, VBZ, JJ, NNS, IN, NN]","[ORG, PERSON]","[NN, RB, VBZ, JJ, NNS, IN, NN, ORG, PERSON]"
126284,IRS Fines Marijuana Merchants For Refusing To ...,"[NNS, NNS, VBP, NNS, IN, VBG, TO, VB, DT, NN]","[ORG, PERSON]","[NNS, NNS, VBP, NNS, IN, VBG, TO, VB, DT, NN, ..."
202925,Refinishing Kitchen Cabinets,"[VBG, NN, NNS]",[],"[VBG, NN, NNS]"
198515,Street Style Face-Off: NYC & Los Angeles Battl...,"[NN, NN, NN, IN, JJ, NNS, NN, IN, JJS, JJ, NN,...",[],"[NN, NN, NN, IN, JJ, NNS, NN, IN, JJS, JJ, NN,..."
63867,One Artist's Heartbreakingly Perfect Response ...,"[CD, VBZ, RB, JJ, NN, TO, DT, NN, NNS]",[CARDINAL],"[CD, VBZ, RB, JJ, NN, TO, DT, NN, NNS, CARDINAL]"
...,...,...,...,...
81360,"When A Stormtrooper Comes Home, It Gets Real","[WRB, DT, NN, VBZ, VBP, PRP, VBZ, JJ]",[],"[WRB, DT, NN, VBZ, VBP, PRP, VBZ, JJ]"
43046,Diesel Trolls Trump's Border Wall With New Cam...,"[NN, NNS, NNS, VBP, NN, IN, JJ, NN]",[PERSON],"[NN, NNS, NNS, VBP, NN, IN, JJ, NN, PERSON]"
93027,The Public Still Can't See The Eric Garner Gra...,"[DT, NN, RB, JJ, VBP, DT, JJ, NN, JJ, NN, NNS,...",[],"[DT, NN, RB, JJ, VBP, DT, JJ, NN, JJ, NN, NNS,..."
4087,Big Ten Announces It Will Begin College Footba...,"[JJ, NN, VBZ, PRP, MD, VB, NN, NN, NN, IN, DT]",[],"[JJ, NN, VBZ, PRP, MD, VB, NN, NN, NN, IN, DT]"


In [40]:
original_df_train['POS+NER_str'] = original_df_train['POS+NER'].apply(lambda x: ' '.join(x))
original_df_test['POS+NER_str']  = original_df_test['POS+NER'].apply(lambda x: ' '.join(x))

## count vectorizer

In [33]:
vectorizer_POS_NER = CountVectorizer(lowercase=False)

In [41]:
x_train_POS_NER = vectorizer_POS_NER.fit_transform(original_df_train['POS+NER_str'])
x_test_POS_NER= vectorizer_POS_NER.transform(original_df_test['POS+NER_str'])

In [43]:
scaler = StandardScaler(with_mean=False)
x_train_POS_NER_scaled = scaler.fit_transform(x_train_POS_NER)
x_test_POS_NER_scaled  = scaler.transform(x_test_POS_NER)

## TF-IDF vectorizer

In [44]:
vectorizer_POS_NER_tfidf = TfidfVectorizer()
x_train_POS_NER_tfidf = vectorizer_POS_NER_tfidf.fit_transform(original_df_train['POS+NER_str'])
x_test_POS_NER_tfidf = vectorizer_POS_NER_tfidf.transform(original_df_test['POS+NER_str'])

In [45]:
scaler_tfidf = StandardScaler(with_mean=False)
x_train_POS_NER_tfidf_scaled = scaler_tfidf.fit_transform(x_train_POS_NER_tfidf)
x_test_POS_NER_tfidf_scaled  = scaler_tfidf.transform(x_test_POS_NER_tfidf)

## Models

In [ ]:
# results_POS_NER_original={}
# joblib.dump(results_POS_NER_original,'results_POS_NER_original')

['results_POS_NER_original']

In [51]:
results_POS_NER_original=joblib.load('results_POS_NER_original')
results_POS_NER_original

{}

In [52]:
x_train_POS_NER_tfidf_scaled.shape

(167616, 51)

In [53]:
x_train_POS_NER_scaled.shape

(167616, 51)

In [54]:
models_POS_NER={
    "SVC_POS_NER": SVC(class_weight=class_weights_dict),
    "MultinomialNB_POS_NER": MultinomialNB(),
    "MLP" :MLPClassifier(hidden_layer_sizes=(100,50),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
    # 'HMM' :GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
    }

## with countvectorizer

In [55]:
for model_name, model in models_POS_NER.items():
    model.fit(x_train_POS_NER_scaled, y_train)

    y_pred_train = model.predict(x_train_POS_NER_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_POS_NER_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_NER_original[model_name] = accuracy

Results for SVC_POS_NER Training: accuracy=0.23531166475754103
Results for SVC_POS_NERTesting: accuracy=0.20102613053334925
              precision    recall  f1-score   support

           1       0.40      0.27      0.32      7120
           2       0.22      0.45      0.30      3589
           3       0.22      0.33      0.26      3473
           4       0.16      0.33      0.22      1980
           5       0.18      0.32      0.23      1963
           6       0.06      0.17      0.09      1269
           7       0.10      0.37      0.16      1268
           8       0.06      0.16      0.09      1198
           9       0.05      0.19      0.08      1016
          10       0.61      0.07      0.13     19029

    accuracy                           0.20     41905
   macro avg       0.21      0.27      0.19     41905
weighted avg       0.41      0.20      0.20     41905

--------------------------------------------------
Results for MultinomialNB_POS_NER Training: accuracy=0.34825434326

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM 

In [56]:
svd_POS_NER = TruncatedSVD(n_components=51, random_state=42)
x_train_svd_POS_NER = svd_POS_NER.fit_transform(x_train_POS_NER)
x_test_svd_POS_NER = svd_POS_NER.transform(x_test_POS_NER)

scaler = StandardScaler()
x_train_svd_POS_NER = scaler.fit_transform(x_train_svd_POS_NER)
x_test_svd_POS_NER = scaler.transform(x_test_svd_POS_NER)

In [57]:
hmm_POS_NER=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [58]:
lengths_train = [1] * x_train_svd_POS_NER.shape[0]
lengths_test = [1] * x_test_svd_POS_NER.shape[0]

In [59]:
hmm_POS_NER.fit(x_train_svd_POS_NER)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [64]:
y_pred_POS_NER_original=hmm_POS_NER.predict(x_train_svd_POS_NER,lengths=lengths_train)
accuracy_hmm_POS_NER_original = accuracy_score(y_train, y_pred_POS_NER_original)   
print(f"training HMM POS_NER Accuracy : {accuracy_hmm_POS_NER_original}")

training HMM POS_NER Accuracy : 0.16991814623902252


In [61]:
y_pred_POS_NER_original=hmm_POS_NER.predict(x_test_svd_POS_NER,lengths=lengths_test)
accuracy_hmm_POS_NER_original = accuracy_score(y_test, y_pred_POS_NER_original)   
print(f"testing HMM POS_NER Accuracy with length : {accuracy_hmm_POS_NER_original}")

testing HMM POS_NER Accuracy with length : 0.1699081255220141


In [62]:
y_pred_POS_NER_original=hmm_POS_NER.predict(x_test_svd_POS_NER,)
accuracy_hmm_POS_NER_original = accuracy_score(y_test, y_pred_POS_NER_original)   
print(f"testing HMM POS_NER Accuracy : {accuracy_hmm_POS_NER_original}")

testing HMM POS_NER Accuracy : 0.1274788211430617


In [63]:
results_POS_NER_original['HMM']=0.1699081255220141

In [65]:
results_POS_NER_original

{'SVC_POS_NER': 0.20102613053334925,
 'MultinomialNB_POS_NER': 0.34726166328600405,
 'MLP': 0.46829733921966354,
 'HMM': 0.1699081255220141}

## With tf-idf

In [67]:
for model_name, model in models_POS_NER.items():
    model.fit(x_train_POS_NER_tfidf_scaled, y_train)

    y_pred_train = model.predict(x_train_POS_NER_tfidf_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_POS_NER_tfidf_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_POS_NER_original["(tf-idf) "+model_name] = accuracy

Results for SVC_POS_NER Training: accuracy=0.2212676594119893
Results for SVC_POS_NER Testing: accuracy=0.19439207731774252
              precision    recall  f1-score   support

           1       0.39      0.25      0.31      7120
           2       0.21      0.44      0.28      3589
           3       0.21      0.33      0.26      3473
           4       0.15      0.34      0.21      1980
           5       0.17      0.33      0.23      1963
           6       0.06      0.17      0.09      1269
           7       0.11      0.33      0.16      1268
           8       0.06      0.15      0.09      1198
           9       0.05      0.19      0.08      1016
          10       0.60      0.07      0.12     19029

    accuracy                           0.19     41905
   macro avg       0.20      0.26      0.18     41905
weighted avg       0.40      0.19      0.19     41905

--------------------------------------------------
Results for MultinomialNB_POS_NER Training: accuracy=0.33820756968

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM

In [68]:
svd_POS_NER_tfidf = TruncatedSVD(n_components=51, random_state=42)
x_train_svd_POS_NER_tfidf = svd_POS_NER_tfidf.fit_transform(x_train_POS_NER_tfidf)
x_test_svd_POS_NER_tfidf = svd_POS_NER_tfidf.transform(x_test_POS_NER_tfidf)

scaler = StandardScaler()
x_train_svd_POS_NER_tfidf_scaled = scaler.fit_transform(x_train_svd_POS_NER_tfidf)
x_test_svd_POS_NER_tfidf_scaled = scaler.transform(x_test_svd_POS_NER_tfidf)

In [69]:
hmm_POS_NER_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [70]:
lengths_train = [1] * x_train_svd_POS_NER_tfidf_scaled.shape[0]
lengths_test = [1] * x_test_svd_POS_NER_tfidf_scaled.shape[0]

In [71]:
hmm_POS_NER_tfidf.fit(x_train_svd_POS_NER_tfidf_scaled)

Model is not converging.  Current: 1433346.4772000897 is not greater than 1433346.5269996263. Delta is -0.049799536587670445


GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [87]:
y_pred_POS_NER_original_tfidf=hmm_POS_NER_tfidf.predict(x_train_svd_POS_NER_tfidf_scaled,)
accuracy_hmm_POS_NER_original = accuracy_score(y_train, y_pred_POS_NER_original_tfidf)   
print(f"training HMM POS_NER Accuracy : {accuracy_hmm_POS_NER_original}")

training HMM POS_NER Accuracy : 0.02867864642993509


In [73]:
y_pred_POS_NER_original_tfidf=hmm_POS_NER_tfidf.predict(x_test_svd_POS_NER_tfidf_scaled,lengths=lengths_test)
accuracy_hmm_POS_NER_original = accuracy_score(y_test, y_pred_POS_NER_original_tfidf)   
print(f"testing HMM POS_NER Accuracy : {accuracy_hmm_POS_NER_original}")

testing HMM POS_NER Accuracy : 0.010213578331941297


In [74]:
y_pred_POS_NER_original_tfidf=hmm_POS_NER_tfidf.predict(x_test_svd_POS_NER_tfidf_scaled,)
accuracy_hmm_POS_NER_original = accuracy_score(y_test, y_pred_POS_NER_original_tfidf)   
print(f"testing HMM POS_NER Accuracy : {accuracy_hmm_POS_NER_original}")

testing HMM POS_NER Accuracy : 0.02789643240663405


In [88]:
results_POS_NER_original['(tf-idf) HMM']=0.010213578331941297

## Save results

In [89]:
results_POS_NER_original

{'SVC_POS_NER': 0.20102613053334925,
 'MultinomialNB_POS_NER': 0.34726166328600405,
 'MLP': 0.46829733921966354,
 'HMM': 0.1699081255220141,
 '(tf-idf) SVC_POS_NER': 0.19439207731774252,
 '(tf-idf) MultinomialNB_POS_NER': 0.3393151175277413,
 '(tf-idf) MLP': 0.4672950721870898,
 '(tf-idf) HMM': 0.010213578331941297}

In [90]:
joblib.dump(results_POS_NER_original,'results_POS_NER_original')

['results_POS_NER_original']

In [ ]:
results_POS_NER_original=joblib.load('results_POS_NER_original')
results_POS_NER_original

{'SVC_NER StopWords removed': 0.14513781171697887,
 'MultinomialNB_NER StopWords removed': 0.42605894284691564,
 'MLP StopWords removed': 0.4576542178737621,
 'HMM StopWords removed': 0.05507695979000119,
 'SVC_NER StopWords kept': 0.1780694427872569,
 'MultinomialNB_NER StopWords kept': 0.39403412480610905,
 'MLP StopWords kept': 0.4642644075885932,
 'HMM StopWords kept': 0.1699081255220141,
 '(tf-idf) SVC_NER StopWords removed': 0.1787853478105238,
 '(tf-idf) MultinomialNB_NER StopWords removed': 0.39792387543252594,
 '(tf-idf) MLP StopWords removed': 0.46373941057153084,
 '(tf-idf) HMM StopWords removed': 0.04474406395418208,
 '(tf-idf) SVC_NER StopWords kept': 0.1787853478105238,
 '(tf-idf) MultinomialNB_NER StopWords kept': 0.39792387543252594,
 '(tf-idf) MLP StopWords kept': 0.46373941057153084,
 '(tf-idf) HMM StopWords kept': 0.04474406395418208}

In [91]:
with open("results_POS_NER_original.txt", "w", encoding="utf-8") as f:
    for key, value in results_POS_NER_original.items():
        f.write(f"{key}: {value}\n")

# ______________________________________________________________________________________

# resampled Data (undersample)

In [92]:
results_POS_NER_undersample={}
# results_POS_NER_undersample=joblib.load('results_POS_NER_undersample')
# results_POS_NER_undersample

## POS_NER feature Extraction + Scaling

In [93]:
undersample_df_train['POS']=undersample_df_train['text'].apply(pos_features)
undersample_df_test['POS']=undersample_df_test['text'].apply(pos_features)
undersample_df_train['NER']=undersample_df_train['text'].apply(NER_features)
undersample_df_test['NER']=undersample_df_test['text'].apply(NER_features)

In [94]:
undersample_df_train['POS+NER']=undersample_df_train['POS']+undersample_df_train['NER']
undersample_df_test['POS+NER']=undersample_df_test['POS']+undersample_df_test['NER']

In [95]:
undersample_df_train['POS+NER_str'] = undersample_df_train['POS+NER'].apply(lambda x: ' '.join(x))
undersample_df_test['POS+NER_str']  = undersample_df_test['POS+NER'].apply(lambda x: ' '.join(x))

### with countvectorizer

In [96]:
vectorizer_POS_NER_undersample = CountVectorizer()
x_train_POS_NER_undersample = vectorizer_POS_NER_undersample.fit_transform(undersample_df_train['POS+NER_str'])
x_test_POS_NER_undersample= vectorizer_POS_NER_undersample.transform(undersample_df_test['POS+NER_str'])

In [97]:
scaler = StandardScaler(with_mean=False)
x_train_POS_NER_undersample_scaled = scaler.fit_transform(x_train_POS_NER_undersample)
x_test_POS_NER_undersample_scaled  = scaler.transform(x_test_POS_NER_undersample)

In [98]:
x_train_POS_NER_undersample_scaled.shape,x_test_POS_NER_undersample_scaled.shape

((66774, 51), (16694, 51))

### with tf-idf vectorizer

In [99]:
vectorizer_POS_NER_undersample_tfidf = TfidfVectorizer()
x_train_POS_NER_undersample_tfidf = vectorizer_POS_NER_undersample_tfidf.fit_transform(undersample_df_train['POS+NER_str'])
x_test_POS_NER_undersample_tfidf= vectorizer_POS_NER_undersample_tfidf.transform(undersample_df_test['POS+NER_str'])

In [100]:
scaler_tfidf = StandardScaler(with_mean=False)
x_train_POS_NER_undersample_scaled_tfidf = scaler_tfidf.fit_transform(x_train_POS_NER_undersample_tfidf)
x_test_POS_NER_undersample_scaled_tfidf  = scaler_tfidf.transform(x_test_POS_NER_undersample_tfidf)

In [101]:
x_train_POS_NER_undersample_scaled_tfidf.shape,x_test_POS_NER_undersample_scaled_tfidf.shape

((66774, 51), (16694, 51))

## Models

In [102]:
models_POS_NER_undersample={
    'SVM': SVC(),
    'MultinomialNB': MultinomialNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(102,51),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
}

## with countvectorizer

In [103]:
for model_name, model in models_POS_NER_undersample.items():
    model.fit(x_train_POS_NER_undersample_scaled, y_train_undersampled)

    y_pred_train = model.predict(x_train_POS_NER_undersample_scaled)
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_POS_NER_undersample_scaled)
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    print('-'*50)
    results_POS_NER_undersample['undersample '+model_name] = accuracy

Results for SVM Training: accuracy=0.32825650702369186
Results for SVM Testing: accuracy=0.29046363963100513
              precision    recall  f1-score   support

           1       0.29      0.36      0.32      2000
           2       0.29      0.58      0.39      2000
           3       0.29      0.40      0.34      2000
           4       0.30      0.41      0.35      1980
           5       0.35      0.37      0.36      1963
           6       0.26      0.04      0.07      1269
           7       0.35      0.15      0.21      1268
           8       0.27      0.03      0.05      1198
           9       0.32      0.03      0.05      1016
          10       0.18      0.17      0.18      2000

    accuracy                           0.29     16694
   macro avg       0.29      0.25      0.23     16694
weighted avg       0.29      0.29      0.26     16694

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.22200257585287686
Results for Mult

### HMM

In [104]:
hmm_POS_NER_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,35)

In [105]:
svd_POS_NER = TruncatedSVD(n_components=51, random_state=42)
x_train_svd_POS_NER_undersample = svd_POS_NER.fit_transform(x_train_POS_NER_undersample_scaled)
x_test_svd_POS_NER_undersample = svd_POS_NER.transform(x_test_POS_NER_undersample_scaled)

scaler = StandardScaler()
x_train_svd_POS_NER_undersample = scaler.fit_transform(x_train_svd_POS_NER_undersample)
x_test_svd_POS_NER_undersample = scaler.transform(x_test_svd_POS_NER_undersample)

#### apply HMM

In [106]:
lengths_train = [1] * x_train_svd_POS_NER_undersample.shape[0]
lengths_test = [1] * x_test_svd_POS_NER_undersample.shape[0]

In [107]:
hmm_POS_NER_undersample.fit(x_train_svd_POS_NER_undersample)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [108]:
y_pred_stem_undersample=hmm_POS_NER_undersample.predict(x_train_svd_POS_NER_undersample,lengths=lengths_train)
accuracy_hmm_stem_undersample = accuracy_score(y_train_undersampled, y_pred_stem_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

training HMM BoW Accuracy with lengths: 0.0


In [109]:
y_pred_stem_undersample=hmm_POS_NER_undersample.predict(x_test_svd_POS_NER_undersample,lengths=lengths_test)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy with lengths: 0.0


In [110]:
y_pred_stem_undersample=hmm_POS_NER_undersample.predict(x_test_svd_POS_NER_undersample)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy: 0.06259734036180664


In [111]:
results_POS_NER_undersample['undersample HMM']=0.06259734036180664

## with tf-idf

In [112]:
for model_name, model in models_POS_NER_undersample.items():
    model.fit(x_train_POS_NER_undersample_scaled_tfidf, y_train_undersampled)

    y_pred_train = model.predict(x_train_POS_NER_undersample_scaled_tfidf)
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_POS_NER_undersample_scaled_tfidf)
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    print('-'*50)
    results_POS_NER_undersample['(tf-idf) undersample '+model_name] = accuracy

Results for SVM Training: accuracy=0.3064965405696828
Results for SVM Testing: accuracy=0.28531208817539233
              precision    recall  f1-score   support

           1       0.28      0.35      0.31      2000
           2       0.28      0.56      0.37      2000
           3       0.29      0.41      0.34      2000
           4       0.30      0.41      0.34      1980
           5       0.34      0.35      0.34      1963
           6       0.28      0.03      0.05      1269
           7       0.36      0.14      0.20      1268
           8       0.30      0.02      0.04      1198
           9       0.33      0.03      0.06      1016
          10       0.19      0.17      0.18      2000

    accuracy                           0.29     16694
   macro avg       0.29      0.25      0.23     16694
weighted avg       0.29      0.29      0.25     16694

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.2211938778566508
Results for Multin

### HMM

In [113]:
hmm_POS_NER_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,18)

In [114]:
svd_POS_NER = TruncatedSVD(n_components=51, random_state=42)
x_train_svd_POS_NER_undersample_tfidf = svd_POS_NER.fit_transform(x_train_POS_NER_undersample_scaled_tfidf)
x_test_svd_POS_NER_undersample_tfidf = svd_POS_NER.transform(x_test_POS_NER_undersample_scaled_tfidf)

scaler = StandardScaler()
x_train_svd_POS_NER_undersample_tfidf = scaler.fit_transform(x_train_svd_POS_NER_undersample_tfidf)
x_test_svd_POS_NER_undersample_tfidf = scaler.transform(x_test_svd_POS_NER_undersample_tfidf)

#### apply HMM

In [115]:
lengths_train = [1]*x_train_svd_POS_NER_undersample_tfidf.shape[0]
lengths_test = [1]*x_test_svd_POS_NER_undersample_tfidf.shape[0]

In [116]:
hmm_POS_NER_undersample_tfidf.fit(x_train_svd_POS_NER_undersample_tfidf)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [121]:
y_pred_stem_tfidf_undersample=hmm_POS_NER_undersample_tfidf.predict(x_train_svd_POS_NER_undersample_tfidf,lengths=lengths_train)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_stem_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_stem_tfidf_undersample}")

training HMM BoW Accuracy : 0.12006170066193429


In [118]:
y_pred_stem_tfidf_undersample=hmm_POS_NER_undersample_tfidf.predict(x_test_svd_POS_NER_undersample_tfidf,lengths=lengths_test)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy with lengths: 0.11956391517910626


In [119]:
y_pred_stem_tfidf_undersample=hmm_POS_NER_undersample_tfidf.predict(x_test_svd_POS_NER_undersample_tfidf)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy: 0.07290044327303223


In [120]:
results_POS_NER_undersample['(tf-idf)undersample HMM']=0.11956391517910626

## save the results

In [122]:
joblib.dump(results_POS_NER_undersample,'results_POS_NER_undersample')
results_POS_NER_undersample

{'undersample SVM': 0.29046363963100513,
 'undersample MultinomialNB': 0.22648855876362764,
 'undersample MLP': 0.2847130705642746,
 'undersample HMM': 0.06259734036180664,
 '(tf-idf) undersample SVM': 0.28531208817539233,
 '(tf-idf) undersample MultinomialNB': 0.22696777285252187,
 '(tf-idf) undersample MLP': 0.28051994728645024,
 '(tf-idf)undersample HMM': 0.11956391517910626}

In [123]:
with open("results_POS_NER_undersample.txt", "w", encoding="utf-8") as f:
    for key, value in results_POS_NER_undersample.items():
        f.write(f"{key}: {value}\n")

# ______________________________________________________________________________________

# resampled Data (rosrus)

In [151]:
results_POS_NER_rosrus={}

## POS_NER feature Extraction + scaling

In [140]:
rosrus_df_train['POS']=rosrus_df_train['text'].apply(pos_features)
rosrus_df_test['POS']=rosrus_df_test['text'].apply(pos_features)
rosrus_df_train['NER']=rosrus_df_train['text'].apply(NER_features)
rosrus_df_test['NER']=rosrus_df_test['text'].apply(NER_features)

In [141]:
rosrus_df_train['POS+NER']=rosrus_df_train['POS']+rosrus_df_train['NER']
rosrus_df_test['POS+NER']=rosrus_df_test['POS']+rosrus_df_test['NER']

In [142]:
rosrus_df_train['POS+NER_str'] = rosrus_df_train['POS+NER'].apply(lambda x: ' '.join(x))
rosrus_df_test['POS+NER_str']  = rosrus_df_test['POS+NER'].apply(lambda x: ' '.join(x))

### with countvictorizer

In [143]:
vectorizer_POS_NER_rosrus = CountVectorizer()
x_train_POS_NER_rosrus = vectorizer_POS_NER_rosrus.fit_transform(rosrus_df_train['POS+NER_str'])
x_test_POS_NER_rosrus= vectorizer_POS_NER_rosrus.transform(rosrus_df_test['POS+NER_str'])

In [144]:
scaler_rosrus = StandardScaler(with_mean=False)
x_train_POS_NER_rosrus_scaled = scaler_rosrus.fit_transform(x_train_POS_NER_rosrus)
x_test_POS_NER_rosrus_scaled  = scaler_rosrus.transform(x_test_POS_NER_rosrus)

In [145]:
x_train_POS_NER_rosrus_scaled.shape,x_test_POS_NER_rosrus_scaled.shape

((130000, 51), (41905, 51))

### with tf-idf

In [146]:
vectorizer_POS_NER_rosrus_tfidf = TfidfVectorizer()
x_train_POS_NER_rosrus_tfidf = vectorizer_POS_NER_rosrus_tfidf.fit_transform(rosrus_df_train['POS+NER_str'])
x_test_POS_NER_rosrus_tfidf= vectorizer_POS_NER_rosrus_tfidf.transform(rosrus_df_test['POS+NER_str'])

In [147]:
scaler_rosrus_tfidf = StandardScaler(with_mean=False)
x_train_POS_NER_rosrus_tfidf_scaled = scaler_rosrus_tfidf.fit_transform(x_train_POS_NER_rosrus_tfidf)
x_test_POS_NER_rosrus_tfidf_scaled  = scaler_rosrus_tfidf.transform(x_test_POS_NER_rosrus_tfidf)

In [148]:
x_train_POS_NER_rosrus_tfidf_scaled.shape,x_test_POS_NER_rosrus_tfidf_scaled.shape

((130000, 51), (41905, 51))

## Models

In [149]:
models_POS_NER_rosrus={
    'SVM': SVC(),
    'MultinomialNB': MultinomialNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(102,51),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
}

### with countvectorizer

In [152]:
for model_name, model in models_POS_NER_rosrus.items():
    model.fit(x_train_POS_NER_rosrus_scaled, y_train_bal)

    y_pred_train = model.predict(x_train_POS_NER_rosrus_scaled)
    accuracy_train=accuracy_score(y_train_bal, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_POS_NER_rosrus_scaled)
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_POS_NER_rosrus['rorus '+model_name] = accuracy

Results for SVM Training: accuracy=0.3471076923076923
Results for SVM Testing: accuracy=0.20221930557212744
              precision    recall  f1-score   support

           1       0.39      0.26      0.31      7120
           2       0.22      0.44      0.29      3589
           3       0.22      0.33      0.26      3473
           4       0.16      0.33      0.22      1980
           5       0.18      0.31      0.23      1963
           6       0.06      0.17      0.09      1269
           7       0.10      0.37      0.16      1268
           8       0.06      0.17      0.09      1198
           9       0.05      0.20      0.09      1016
          10       0.62      0.08      0.14     19029

    accuracy                           0.20     41905
   macro avg       0.21      0.27      0.19     41905
weighted avg       0.41      0.20      0.20     41905

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.19784615384615384
Results for Multi

#### HMM

In [153]:
hmm_POS_NER_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

##### dimension reduction (,35)

In [154]:
svd_POS_NER = TruncatedSVD(n_components=51, random_state=42)
x_train_svd_POS_NER_rosrus = svd_POS_NER.fit_transform(x_train_POS_NER_rosrus_scaled)
x_test_svd_POS_NER_rosrus = svd_POS_NER.transform(x_test_POS_NER_rosrus_scaled)

scaler = StandardScaler()
x_train_svd_POS_NER_rosrus = scaler.fit_transform(x_train_svd_POS_NER_rosrus)
x_test_svd_POS_NER_rosrus = scaler.transform(x_test_svd_POS_NER_rosrus)

In [155]:
lengths_train = [1]*x_train_svd_POS_NER_rosrus.shape[0]
lengths_test = [1]*x_test_svd_POS_NER_rosrus.shape[0]

In [156]:
hmm_POS_NER_rosrus.fit(x_train_svd_POS_NER_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [160]:
y_pred_POS_NER_rosrus=hmm_POS_NER_rosrus.predict(x_train_svd_POS_NER_rosrus,)
accuracy_hmm_POS_NER_rosrus = accuracy_score(y_train_bal, y_pred_POS_NER_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_POS_NER_rosrus}")

training HMM BoW Accuracy with lengths: 0.05407692307692308


In [158]:
y_pred_POS_NER_rosrus=hmm_POS_NER_rosrus.predict(x_test_svd_POS_NER_rosrus,lengths=lengths_test)
accuracy_hmm_POS_NER_rosrus = accuracy_score(y_test_bal, y_pred_POS_NER_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_POS_NER_rosrus}")

testing HMM BoW Accuracy with lengths: 0.0


In [159]:
y_pred_POS_NER_rosrus=hmm_POS_NER_rosrus.predict(x_test_svd_POS_NER_rosrus)
accuracy_hmm_POS_NER_rosrus = accuracy_score(y_test_bal, y_pred_POS_NER_rosrus)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_POS_NER_rosrus}")

testing HMM BoW Accuracy: 0.019782842142942368


In [161]:
results_POS_NER_rosrus['rosrus HMM ']=0.019782842142942368

### with tf-idf

In [162]:
for model_name, model in models_POS_NER_rosrus.items():
    model.fit(x_train_POS_NER_rosrus_tfidf_scaled, y_train_bal)

    y_pred_train = model.predict(x_train_POS_NER_rosrus_tfidf_scaled)
    accuracy_train=accuracy_score(y_train_bal, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_POS_NER_rosrus_tfidf_scaled)
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_POS_NER_rosrus['(tf-idf) rorus '+model_name] = accuracy

Results for SVM Training: accuracy=0.3200076923076923
Results for SVM Testing: accuracy=0.19303185777353538
              precision    recall  f1-score   support

           1       0.38      0.25      0.30      7120
           2       0.21      0.44      0.29      3589
           3       0.21      0.33      0.25      3473
           4       0.15      0.34      0.21      1980
           5       0.17      0.32      0.22      1963
           6       0.06      0.16      0.08      1269
           7       0.11      0.33      0.16      1268
           8       0.06      0.16      0.09      1198
           9       0.05      0.19      0.08      1016
          10       0.61      0.07      0.12     19029

    accuracy                           0.19     41905
   macro avg       0.20      0.26      0.18     41905
weighted avg       0.40      0.19      0.18     41905

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.1993
Results for MultinomialNB Test

#### HMM

In [163]:
hmm_POS_NER_rosrus_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

##### dimension reduction (,35)

In [164]:
svd_POS_NER_tfidf = TruncatedSVD(n_components=51, random_state=42)
x_train_svd_POS_NER_tfidf_rosrus = svd_POS_NER_tfidf.fit_transform(x_train_POS_NER_rosrus_tfidf_scaled)
x_test_svd_POS_NER_tfidf_rosrus = svd_POS_NER_tfidf.transform(x_test_POS_NER_rosrus_tfidf_scaled)

scaler = StandardScaler()
x_train_svd_POS_NER_tfidf_rosrus = scaler.fit_transform(x_train_svd_POS_NER_tfidf_rosrus)
x_test_svd_POS_NER_tfidf_rosrus = scaler.transform(x_test_svd_POS_NER_tfidf_rosrus)

In [165]:
lengths_train = [1]*x_train_svd_POS_NER_tfidf_rosrus.shape[0]
lengths_test = [1]*x_test_svd_POS_NER_tfidf_rosrus.shape[0]

In [166]:
hmm_POS_NER_rosrus_tfidf.fit(x_train_svd_POS_NER_tfidf_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [167]:
y_pred_POS_NER_rosrus_tfidf=hmm_POS_NER_rosrus_tfidf.predict(x_train_svd_POS_NER_tfidf_rosrus,lengths=lengths_train)
accuracy_hmm_POS_NER_rosrus_tfidf = accuracy_score(y_train_bal, y_pred_POS_NER_rosrus_tfidf)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_POS_NER_rosrus_tfidf}")

training HMM BoW Accuracy with lengths: 0.0940076923076923


In [168]:
y_pred_POS_NER_rosrus_tfidf=hmm_POS_NER_rosrus_tfidf.predict(x_test_svd_POS_NER_tfidf_rosrus,lengths=lengths_test)
accuracy_hmm_POS_NER_rosrus_tfidf = accuracy_score(y_test_bal, y_pred_POS_NER_rosrus_tfidf)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_POS_NER_rosrus_tfidf}")

testing HMM BoW Accuracy with lengths: 0.14757188879608638


In [169]:
y_pred_POS_NER_rosrus_tfidf=hmm_POS_NER_rosrus_tfidf.predict(x_test_svd_POS_NER_tfidf_rosrus,)
accuracy_hmm_POS_NER_rosrus_tfidf = accuracy_score(y_test_bal, y_pred_POS_NER_rosrus_tfidf)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_POS_NER_rosrus_tfidf}")

testing HMM BoW Accuracy: 0.09791194368213817


In [171]:
results_POS_NER_rosrus['(tf-idf) rosrus HMM']=0.14757188879608638

In [172]:
results_POS_NER_rosrus

{'rorus SVM': 0.20221930557212744,
 'rorus MultinomialNB': 0.14520940221930556,
 'rorus MLP': 0.17176947858250805,
 'rosrus HMM ': 0.019782842142942368,
 '(tf-idf) rorus SVM': 0.19303185777353538,
 '(tf-idf) rorus MultinomialNB': 0.14494690371077437,
 '(tf-idf) rorus MLP': 0.15661615559002506,
 '(tf-idf) rosrus HMM': 0.14757188879608638}

## Save results

In [173]:
joblib.dump(results_POS_NER_rosrus,'results_POS_NER_rosrus')

['results_POS_NER_rosrus']

In [174]:
results_POS_NER_rosrus=joblib.load('results_POS_NER_rosrus')
results_POS_NER_rosrus

{'rorus SVM': 0.20221930557212744,
 'rorus MultinomialNB': 0.14520940221930556,
 'rorus MLP': 0.17176947858250805,
 'rosrus HMM ': 0.019782842142942368,
 '(tf-idf) rorus SVM': 0.19303185777353538,
 '(tf-idf) rorus MultinomialNB': 0.14494690371077437,
 '(tf-idf) rorus MLP': 0.15661615559002506,
 '(tf-idf) rosrus HMM': 0.14757188879608638}

In [175]:
with open("results_POS_NER_rosrus.txt", "w", encoding="utf-8") as f:
    for key, value in results_POS_NER_rosrus.items():
        f.write(f"{key}: {value}\n")